# 🇴🇲 Oman Ministry of Health - Annual Health Reports

This notebook demonstrates how to access health statistics from the **Sultanate of Oman Ministry of Health (MOH)** Annual Health Reports.

**Data Sources:**
- **Annual Health Reports** - Comprehensive health statistics (1983/84–present)
- **Morbidity and Mortality Data** - Disease-specific case counts and deaths (Chapter 9)
- **Health Status Indicators** - Life expectancy, infant mortality, immunization (Chapter 2)
- **Health Service Utilization** - Outpatient visits, hospital admissions, bed occupancy (Chapter 7)
- **Human Resources for Health** - Healthcare workforce statistics (Chapter 4)
- **Governorate-level Data** - Subnational health statistics by region

**Coverage:**
- **Geographic**: 11 governorates (Wilayat)
- **Temporal**: 1983/1984 to present (annual)
- **Diseases**: Malaria, TB, diabetes, CVD, injuries, and more

**Requirements:**
```bash
pip install pandas matplotlib seaborn requests beautifulsoup4 pypdf
```

## 1. Setup and Imports

In [ ]:
import sys
import warnings
from datetime import datetime

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
%matplotlib inline

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Import Oman MOH accessor
from epidatasets.sources.oman_moh import OmanMOHAccessor

print("✅ Imports completed successfully!")
print(f"⏰ Current time: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

## 2. Initialize Oman MOH Accessor

In [ ]:
# Initialize the Oman MOH accessor
moh = OmanMOHAccessor()

print("🇴🇲 Oman MOH Accessor initialized")
print(f"Source: {moh.source_name}")
print(f"Portal: {moh.source_url}")

## 3. Explore Governorates

In [ ]:
# List Omani governorates
governorates = moh.GOVERNORATES
print(f"\n🇴🇲 Omani Governorates ({len(governorates)} total):")
for i, gov in enumerate(governorates, 1):
    print(f"  {i}. {gov}")

# Visualize governorates
fig, ax = plt.subplots(figsize=(8, 6))
y_pos = range(len(governorates))
colors = plt.cm.Blues([0.3 + 0.7 * i / len(governorates) for i in range(len(governorates))])
ax.barh(y_pos, [1]*len(governorates), color=colors)
ax.set_yticks(y_pos)
ax.set_yticklabels(governorates, fontsize=10)
ax.set_xlabel('Count')
ax.set_title('Omani Governorates')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 4. List Available Annual Reports

In [ ]:
# List available annual health reports
reports = moh.list_available_reports()
print(f"\n📊 Available Annual Health Reports ({len(reports)} found):")
print(reports.head(10).to_string(index=False))

# Plot reports by year
if not reports.empty and 'year' in reports.columns:
    yearly_counts = reports['year'].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(10, 4))
    yearly_counts.plot(kind='bar', ax=ax, color='teal')
    ax.set_xlabel('Year')
    ax.set_ylabel('Number of Reports')
    ax.set_title('Oman MOH Annual Health Reports by Year')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

    print(f"\n📈 Report Coverage: {reports['year'].min()} to {reports['year'].max()}")

## 5. Download an Annual Health Report

In [ ]:
# Download a specific annual report
try:
    report = moh.get_annual_report(year=2023)
    print(f"\n📄 Annual Report downloaded:")
    print(f"  Year: {report['year']}")
    print(f"  PDF Path: {report['pdf_path']}")
    print(f"  URL: {report['url']}")
    print(f"  File exists: {report['pdf_path'].exists() if hasattr(report['pdf_path'], 'exists') else 'N/A'}")

    # Extract text from PDF
    text = moh._extract_pdf_text(report['pdf_path'])
    print(f"\n📝 Extracted Text (first 1000 chars):")
    print(text[:1000] if text else "No text extracted")
except Exception as e:
    print(f"⚠️ Could not download report: {e}")
    print("This may require network access or the specific report may not be available.")

## 6. Extract Morbidity and Mortality Data

In [ ]:
# Extract morbidity and mortality data from the report
try:
    morbidity = moh.extract_morbidity_mortality(report['pdf_path'])
    print(f"\n🦠 Morbidity and Mortality Data ({len(morbidity)} diseases):")
    if not morbidity.empty:
        print(morbidity.head(15).to_string(index=False))

        # Visualize top diseases
        if 'cases' in morbidity.columns:
            top_diseases = morbidity.nlargest(10, 'cases')
            fig, ax = plt.subplots(figsize=(10, 6))
            ax.barh(top_diseases['disease'], top_diseases['cases'], color='crimson')
            ax.set_xlabel('Number of Cases')
            ax.set_title('Top 10 Diseases by Case Count (Oman 2023)')
            ax.invert_yaxis()
            plt.tight_layout()
            plt.show()
    else:
        print("  No morbidity data extracted.")
except Exception as e:
    print(f"⚠️ Could not extract morbidity data: {e}")

## 7. Extract Health Status Indicators

In [ ]:
# Extract health status indicators
try:
    indicators = moh.extract_health_indicators(report['pdf_path'])
    print(f"\n📈 Health Status Indicators ({len(indicators)} found):")
    if not indicators.empty:
        print(indicators.to_string(index=False))

        # Visualize key indicators
        if 'value' in indicators.columns and 'unit' in indicators.columns:
            # Filter numeric indicators
            numeric = indicators[indicators['value'].apply(lambda x: isinstance(x, (int, float)))].copy()
            if not numeric.empty:
                fig, ax = plt.subplots(figsize=(10, 6))
                ax.barh(numeric['indicator'], numeric['value'], color='seagreen')
                ax.set_xlabel('Value')
                ax.set_title('Oman Health Status Indicators (2023)')
                ax.invert_yaxis()
                plt.tight_layout()
                plt.show()
    else:
        print("  No health indicators extracted.")
except Exception as e:
    print(f"⚠️ Could not extract health indicators: {e}")

## 8. Extract Health Service Utilization

In [ ]:
# Extract health service utilization data
try:
    utilization = moh.extract_health_utilization(report['pdf_path'])
    print(f"\n🏥 Health Service Utilization ({len(utilization)} metrics):")
    if not utilization.empty:
        print(utilization.to_string(index=False))

        # Visualize utilization metrics
        if 'value' in utilization.columns:
            numeric = utilization[utilization['value'].apply(lambda x: isinstance(x, (int, float)))].copy()
            if not numeric.empty:
                fig, ax = plt.subplots(figsize=(10, 5))
                ax.barh(numeric['metric'], numeric['value'], color='steelblue')
                ax.set_xlabel('Value')
                ax.set_title('Oman Health Service Utilization (2023)')
                ax.invert_yaxis()
                plt.tight_layout()
                plt.show()
    else:
        print("  No utilization data extracted.")
except Exception as e:
    print(f"⚠️ Could not extract utilization data: {e}")

## 9. Governorate-level Health Data

In [ ]:
# Extract governorate-level data
try:
    gov_data = moh.get_governorate_data(report['pdf_path'])
    print(f"\n🗺️ Governorate-level Data ({len(gov_data)} governorates):")
    if not gov_data.empty:
        print(gov_data.to_string(index=False))

        # Visualize by governorate
        if 'value' in gov_data.columns:
            numeric = gov_data[gov_data['value'].apply(lambda x: isinstance(x, (int, float)))].copy()
            if not numeric.empty:
                fig, ax = plt.subplots(figsize=(10, 6))
                ax.barh(numeric['governorate'], numeric['value'], color='darkorange')
                ax.set_xlabel('Value')
                ax.set_title('Oman Health Data by Governorate (2023)')
                ax.invert_yaxis()
                plt.tight_layout()
                plt.show()
    else:
        print("  No governorate data extracted.")
except Exception as e:
    print(f"⚠️ Could not extract governorate data: {e}")

## 10. Compare Multiple Years

In [ ]:
# Compare health indicators across multiple years
try:
    years = [2020, 2021, 2022, 2023]
    all_indicators = []

    for year in years:
        try:
            r = moh.get_annual_report(year=year)
            ind = moh.extract_health_indicators(r['pdf_path'])
            if not ind.empty:
                ind['year'] = year
                all_indicators.append(ind)
        except Exception as ex:
            print(f"  ⚠️ Year {year}: {ex}")

    if all_indicators:
        combined = pd.concat(all_indicators, ignore_index=True)
        print(f"\n📊 Combined indicators across {len(years)} years:")
        print(combined.head(20).to_string(index=False))

        # Plot trends for life expectancy
        le_data = combined[combined['indicator'] == 'life_expectancy']
        if not le_data.empty and 'value' in le_data.columns:
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.plot(le_data['year'], le_data['value'], marker='o', color='green', linewidth=2)
            ax.set_xlabel('Year')
            ax.set_ylabel('Life Expectancy (years)')
            ax.set_title('Oman: Life Expectancy Trend')
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
    else:
        print("\n⚠️ No multi-year data available.")
except Exception as e:
    print(f"⚠️ Could not compare years: {e}")

## 11. Other Statistical Reports

In [ ]:
# List other statistical reports from MOH
try:
    other_reports = moh.get_other_statistical_reports()
    print(f"\n📑 Other Statistical Reports ({len(other_reports)} found):")
    if not other_reports.empty:
        print(other_reports.head(10).to_string(index=False))
    else:
        print("  No other reports available or could not fetch.")
except Exception as e:
    print(f"⚠️ Could not fetch other reports: {e}")

## 12. Summary and Next Steps

This notebook demonstrated how to access Oman Ministry of Health data through the `OmanMOHAccessor`.

**Key capabilities:**
- ✅ Access annual health reports (1983/84–present)
- ✅ Extract morbidity and mortality statistics by disease
- ✅ Retrieve health status indicators (life expectancy, IMR, etc.)
- ✅ Analyze health service utilization patterns
- ✅ Compare data across governorates (Wilayat)
- ✅ Track trends across multiple years

**Potential use cases:**
- Non-communicable disease burden analysis (diabetes, CVD)
- Healthcare system capacity planning
- Regional health disparities assessment
- Integration with climate data for heat-related illness modeling

**Documentation:**
- **MOH Portal**: https://moh.gov.om/en/statistics/annual-health-reports/
- **GHDX Catalog**: https://ghdx.healthdata.org/series/oman-annual-health-report
- **Repository**: https://github.com/fccoelho/epidemiological-datasets